In [1]:
# Cell 1 — Import
import sys
import pandas as pd
sys.path.append('../')
from src.event_log.trace_builder import build_traces
from src.event_log.neo4j_loader import Neo4jLoader

In [2]:
# Cell 2 — Load dataset đã xử lý từ Parquet (nhanh)
df = pd.read_parquet('../data/processed/event_log_clean.parquet')
print(f"Đã load: {df['case_id'].nunique()} case, "
      f"{len(df)} event, {len(df.columns)} cột")

Đã load: 28512 case, 1047482 event, 22 cột


In [3]:
# Cell 3 — Khởi động loader và tạo index
loader = Neo4jLoader(env_path='../.env')
loader.create_indexes()

Kết nối Neo4j: bolt://localhost:7687
Đã tạo 7 index


In [4]:
# Cell 4 — Xoá data cũ
loader.clear_trace_data()

Đã xoá trace data cũ


In [5]:
# Cell 5 — Build traces từ DataFrame
traces = build_traces(df)
print(f"\nVí dụ trace đầu tiên:")
print(f"  case_id: {traces[0]['case_id']}")
print(f"  num_events: {traces[0]['num_events']}")
print(f"  requested_amount: {traces[0]['requested_amount']}")

Đã build 28512 trace

Ví dụ trace đầu tiên:
  case_id: Application_1000086665
  num_events: 22
  requested_amount: 5000.0


In [6]:
# Cell 6 — Test với 10 case trước
print("=== TEST 10 CASE ===")
stats = loader.load_traces(traces, batch_size=5, limit=10)

=== TEST 10 CASE ===
Bắt đầu load 10 trace...
  5/10 trace (OK=5, lỗi=0)
  10/10 trace (OK=10, lỗi=0)

Hoàn thành: 10/10 trace loaded


In [7]:
# Cell 7 — Kiểm tra kết quả test
counts = loader.verify()

Thống kê Neo4j:
  Case                :       10
  ApplicationEvent    :       79
  WorkflowEvent       :      253
  OfferEvent          :       48
  FOLLOWED_BY         :    1,869
  HAS_APP_EVENT       :       79


In [8]:
# Cell 8 — Nếu test OK, load toàn bộ
# Xoá data test trước
loader.clear_trace_data()

print("=== LOAD TOÀN BỘ 28,512 CASE ===")
stats = loader.load_traces(traces, batch_size=100)





Đã xoá trace data cũ
=== LOAD TOÀN BỘ 28,512 CASE ===
Bắt đầu load 28512 trace...
  100/28512 trace (OK=100, lỗi=0)
  200/28512 trace (OK=200, lỗi=0)
  300/28512 trace (OK=300, lỗi=0)
  400/28512 trace (OK=400, lỗi=0)
  500/28512 trace (OK=500, lỗi=0)
  600/28512 trace (OK=600, lỗi=0)
  700/28512 trace (OK=700, lỗi=0)
  800/28512 trace (OK=800, lỗi=0)
  900/28512 trace (OK=900, lỗi=0)
  1000/28512 trace (OK=1000, lỗi=0)
  1100/28512 trace (OK=1100, lỗi=0)
  1200/28512 trace (OK=1200, lỗi=0)
  1300/28512 trace (OK=1300, lỗi=0)
  1400/28512 trace (OK=1400, lỗi=0)
  1500/28512 trace (OK=1500, lỗi=0)
  1600/28512 trace (OK=1600, lỗi=0)
  1700/28512 trace (OK=1700, lỗi=0)
  1800/28512 trace (OK=1800, lỗi=0)
  1900/28512 trace (OK=1900, lỗi=0)
  2000/28512 trace (OK=2000, lỗi=0)
  2100/28512 trace (OK=2100, lỗi=0)
  2200/28512 trace (OK=2200, lỗi=0)
  2300/28512 trace (OK=2300, lỗi=0)
  2400/28512 trace (OK=2400, lỗi=0)
  2500/28512 trace (OK=2500, lỗi=0)
  2600/28512 trace (OK=2600, lỗi=0)


In [9]:
# Cell 9 — Kiểm tra kết quả cuối
counts = loader.verify()
loader.close()

Thống kê Neo4j:
  Case                :   28,512
  ApplicationEvent    :  213,287
  WorkflowEvent       :  661,173
  OfferEvent          :  173,022
  FOLLOWED_BY         : 1,020,469
  HAS_APP_EVENT       :  213,287
Đã đóng kết nối Neo4j


In [10]:
# Cell 10 — Kiểm tra trong Neo4j Browser bằng Python
with loader.driver.session() as session:
    result = session.run("""
        MATCH (c:Case {case_id: 'Application_652823628'})
        -[:HAS_APP_EVENT|HAS_WF_EVENT|HAS_OFFER_EVENT]->(e)
        RETURN labels(e)[0] AS type,
               e.activity_name AS activity,
               e.lifecycle AS lifecycle,
               e.seq_index AS seq
        ORDER BY seq
    """)
    for r in result:
        print(f"  [{r['seq']:>2}] {r['type']:<20} "
              f"{r['activity']:<35} {r['lifecycle']}")

C:\Users\vuxnye\AppData\Local\Temp\ipykernel_15436\4024825476.py:2: DeprecationWarning: Using a driver after it has been closed is deprecated. Future versions of the driver will raise an error.
  with loader.driver.session() as session:


  [ 0] ApplicationEvent     A_Create Application                complete
  [ 1] ApplicationEvent     A_Submitted                         complete
  [ 2] WorkflowEvent        W_Handle leads                      schedule
  [ 3] WorkflowEvent        W_Handle leads                      withdraw
  [ 4] WorkflowEvent        W_Complete application              schedule
  [ 5] ApplicationEvent     A_Concept                           complete
  [ 6] WorkflowEvent        W_Complete application              start
  [ 7] WorkflowEvent        W_Complete application              suspend
  [ 8] ApplicationEvent     A_Accepted                          complete
  [ 9] OfferEvent           O_Create Offer                      complete
  [10] OfferEvent           O_Created                           complete
  [11] OfferEvent           O_Sent (mail and online)            complete
  [12] WorkflowEvent        W_Complete application              ate_abort
  [13] WorkflowEvent        W_Call after offers       